In [ ]:
import pandas as pd
import geopandas as gpd
from matplotlib import pyplot as plt
from matplotlib.patches import Patch
import sys
import os

sys.path.append(os.path.abspath(".."))
from utils.plot_style import apply_plot_style
from utils.config import root_dir

apply_plot_style()

In [ ]:
# load data layers
us_states_gdf = gpd.read_file(
     f"zip://{root_dir}/data_input/state_boundaries/cb_2018_us_state_20m.zip/cb_2018_us_state_20m.shp"
)
study_periods = ["2008_2012", "2013_2017", "2018_2022"]
huc12 = gpd.read_file(root_dir + "results/aoi_huc12_boundaries.gpkg")
huc12_rsei = pd.read_csv(root_dir + "results/huc12_rsei_toxconc_weighted.csv", dtype={"huc12": str})
# combine tabular and spatial data using huc12 ids
huc12 = huc12.merge(huc12_rsei, on="huc12")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(4.5, 5), layout="constrained")

for index, study_period in enumerate(study_periods):
    us_states_gdf.to_crs(huc12.crs).plot(
        ax=axs[index], edgecolor="#adb5bd", facecolor="#f8f9fa", lw=1.0
    )
    huc12.plot(ax=axs[index], color="lightgrey")
    huc12[huc12[f"{study_period}_TOXCONC"].notnull()].plot(
        ax=axs[index], color="#222222"
    )
    axs[index].set_title(study_period.replace("_", " - "))
    axs[index].set_xticks([])
    axs[index].set_yticks([])

# Manually create legend handles
legend_elements = [Patch(facecolor="#222222", label="impacted")]

# Add the legend to the last map
axs[-1].legend(handles=legend_elements)

# Set axis limits to match the aoi bounds exactly
minx, miny, maxx, maxy = huc12.total_bounds
for ax in [0, 1, 2]:
    axs[ax].set_xlim(minx, maxx)
    axs[ax].set_ylim(miny, maxy)


plt.show()

fig.savefig(f"{root_dir}/figures/Figure S3.pdf")